In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.base import BaseEstimator, ClassifierMixin

pd.set_option('display.max_columns', 200)
plt.rcParams['figure.dpi'] = 100

df = pd.read_parquet('../data/loans_clean.parquet')
df['dti'] = df['dti'].replace([-1, 999], np.nan).clip(upper=100)
df['credit_history_months'] = df['credit_history_months'].replace(999, np.nan)
df['annual_inc'] = df['annual_inc'].clip(upper=1_000_000)
df['revol_util'] = df['revol_util'].clip(upper=100)

df_val = df[df['issue_year'] == 2016].copy()
y_val = df_val['default'].values

print(f"Val loans: {len(df_val):,}")

Val loans: 293,095


In [41]:
def prep_features(df):
    df = df.copy()
    redundant = ['fico_range_high', 'funded_amnt', 'funded_amnt_inv',
                 'num_sats', 'installment', 'num_rev_tl_bal_gt_0']
    joint_cols = [c for c in df.columns if c.startswith('sec_app_') or c.endswith('_joint')]
    high_cardinality = ['zip_code', 'sub_grade']
    split_cols = ['issue_year']
    cols_to_drop = redundant + joint_cols + high_cardinality + split_cols
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    
    emp_map = {f'{i} year{"s" if i != 1 else ""}': i for i in range(1, 10)}
    emp_map['< 1 year'] = 0
    emp_map['10+ years'] = 10
    df['emp_length'] = df['emp_length'].map(emp_map)
    return df


df_val_fe = prep_features(df_val)
X_val = df_val_fe.drop(columns=['default'])
X_val['term'] = X_val['term'].str.extract(r'(\d+)').astype(int)

numeric_cols = X_val.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_val.select_dtypes(include=['object']).columns.tolist()

X_val[numeric_cols] = X_val[numeric_cols].astype('float64')
X_val[categorical_cols] = X_val[categorical_cols].astype(str)

# Impute (we need no NaN for batch prediction)
for col in numeric_cols:
    X_val[col] = X_val[col].fillna(X_val[col].median())
for col in categorical_cols:
    X_val[col] = X_val[col].fillna('missing')

print(f"X_val: {X_val.shape}, NaN: {X_val.isna().sum().sum()}")

X_val: (293095, 79), NaN: 0


In [42]:
with open('../models/champion_lgbm.pkl', 'rb') as f:
    champion = pickle.load(f)


class CalibratedChampionClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, preprocessor, model, calibrator, threshold):
        self.preprocessor = preprocessor
        self.model = model
        self.calibrator = calibrator
        self.threshold = threshold
        self.classes_ = np.array([0, 1])
    def fit(self, X, y):
        return self
    def predict_proba(self, X):
        X_proc = self.preprocessor.transform(X)
        raw = self.model.predict_proba(X_proc)[:, 1]
        cal = self.calibrator.predict(raw)
        return np.column_stack([1 - cal, cal])
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= self.threshold).astype(int)


champion_clf = CalibratedChampionClassifier(
    preprocessor=champion['preprocessor'],
    model=champion['base_model'],
    calibrator=champion['calibrator'],
    threshold=champion['best_threshold']
)
best_threshold = champion['best_threshold']

all_probs = champion_clf.predict_proba(X_val)[:, 1]
print(f"Champion loaded. Threshold: {best_threshold:.4f}")
print(f"Approved in val: {(all_probs < best_threshold).sum():,}")
print(f"Rejected in val: {(all_probs >= best_threshold).sum():,}")

/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Champion loaded. Threshold: 0.1457
Approved in val: 99,701
Rejected in val: 193,394


In [43]:
# EASY case: P just above threshold (single-feature fixes likely possible)
easy_mask = (
    (all_probs >= 0.16) & (all_probs <= 0.22) &
    (X_val['grade'].isin(['C', 'D'])) &
    (X_val['term'] == 60)
)
easy_idx = int(np.where(easy_mask)[0][0])

# HARD case: P much higher (will need multiple changes)
hard_mask = (
    (all_probs >= 0.30) & (all_probs <= 0.40) &
    (X_val['grade'].isin(['C', 'D'])) &
    (X_val['term'] == 60)
)
hard_idx = int(np.where(hard_mask)[0][0])

for label, idx in [('EASY (P just above threshold)', easy_idx),
                    ('HARD (P much higher)', hard_idx)]:
    r = X_val.iloc[idx]
    print(f"\n=== {label} — borrower idx {idx} ===")
    print(f"  P(default): {all_probs[idx]:.4f}  (threshold: {best_threshold:.4f})")
    print(f"  Loan:     ${r['loan_amnt']:,.0f}, {r['term']:.0f} months, {r['int_rate']:.1f}% rate, grade {r['grade']}")
    print(f"  Borrower: ${r['annual_inc']:,.0f}/yr, DTI {r['dti']:.1f}%, FICO {r['fico_range_low']:.0f}")
    print(f"  Other:    emp_length={r['emp_length']:.0f} yrs, {r['home_ownership']}, {r['purpose']}")


=== EASY (P just above threshold) — borrower idx 42 ===
  P(default): 0.2092  (threshold: 0.1457)
  Loan:     $12,800, 60 months, 14.5% rate, grade C
  Borrower: $85,000/yr, DTI 9.9%, FICO 715
  Other:    emp_length=10 yrs, MORTGAGE, debt_consolidation

=== HARD (P much higher) — borrower idx 0 ===
  P(default): 0.3128  (threshold: 0.1457)
  Loan:     $10,000, 60 months, 13.5% rate, grade C
  Borrower: $32,000/yr, DTI 13.1%, FICO 750
  Other:    emp_length=3 yrs, RENT, debt_consolidation


In [44]:
def find_single_feature_cfs(query_row, model, feature_specs, threshold, n_steps=100):
    """For each feature, find the smallest change that flips REJECT to APPROVE."""
    results = []
    
    for feature, (low, high) in feature_specs.items():
        original_val = query_row[feature].iloc[0]
        values = np.linspace(low, high, n_steps)
        
        # Batch predict (much faster than looping)
        batch = pd.concat([query_row] * n_steps, ignore_index=True)
        batch[feature] = values
        probs = model.predict_proba(batch)[:, 1]
        
        flips = probs < threshold
        if not flips.any():
            continue
        
        # Smallest change among the flipping values
        changes = np.abs(values - original_val)
        changes[~flips] = np.inf
        best = np.argmin(changes)
        
        results.append({
            'feature':   feature,
            'original':  original_val,
            'new':       values[best],
            'change':    values[best] - original_val,
            'new_prob':  probs[best]
        })
    
    return results


feature_specs = {
    'loan_amnt':            (1000, 40000),
    'int_rate':             (5, 30),
    'dti':                  (0, 60),
    'annual_inc':           (20000, 500000),
    'emp_length':           (0, 10),
    'fico_range_low':       (600, 850),
    'acc_open_past_24mths': (0, 30),
}


def print_cfs(cfs, label, original_prob):
    print(f"\n{'='*80}")
    print(f"{label}: P(default)={original_prob:.4f}, REJECTED")
    print(f"{'='*80}")
    if not cfs:
        print("  No single-feature change can flip this rejection.")
        return
    for cf in cfs:
        if cf['feature'] in ['loan_amnt', 'annual_inc']:
            print(f"  {cf['feature']:25}: ${cf['original']:>10,.0f} -> ${cf['new']:>10,.0f}  "
                  f"({cf['change']:+,.0f})  P -> {cf['new_prob']:.4f}")
        else:
            print(f"  {cf['feature']:25}: {cf['original']:>10.2f} -> {cf['new']:>10.2f}  "
                  f"({cf['change']:+.2f})  P -> {cf['new_prob']:.4f}")


easy_cfs = find_single_feature_cfs(X_val.iloc[[easy_idx]], champion_clf, feature_specs, best_threshold)
print_cfs(easy_cfs, f"EASY CASE (idx {easy_idx})", all_probs[easy_idx])

hard_cfs = find_single_feature_cfs(X_val.iloc[[hard_idx]], champion_clf, feature_specs, best_threshold)
print_cfs(hard_cfs, f"HARD CASE (idx {hard_idx})", all_probs[hard_idx])

/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/nachimora


EASY CASE (idx 42): P(default)=0.2092, REJECTED
  loan_amnt                : $    12,800 -> $     1,788  (-11,012)  P -> 0.1451
  int_rate                 :      14.49 ->       7.53  (-6.96)  P -> 0.1388

HARD CASE (idx 0): P(default)=0.3128, REJECTED
  No single-feature change can flip this rejection.


/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [45]:
def find_pairwise_cfs(query_row, model, feature_specs, threshold, n_per_feature=15):
    """Find pairs of feature changes that flip rejection. Returns sorted by minimum combined change."""
    features = list(feature_specs.keys())
    results = []
    
    for i, f1 in enumerate(features):
        low1, high1 = feature_specs[f1]
        v1_range = np.linspace(low1, high1, n_per_feature)
        orig1 = query_row[f1].iloc[0]
        
        for j, f2 in enumerate(features):
            if j <= i:
                continue
            low2, high2 = feature_specs[f2]
            v2_range = np.linspace(low2, high2, n_per_feature)
            orig2 = query_row[f2].iloc[0]
            
            v1s, v2s = np.meshgrid(v1_range, v2_range)
            n = v1s.size
            
            batch = pd.concat([query_row] * n, ignore_index=True)
            batch[f1] = v1s.flatten()
            batch[f2] = v2s.flatten()
            
            probs = model.predict_proba(batch)[:, 1]
            flips = probs < threshold
            if not flips.any():
                continue
            
            v1_flat = v1s.flatten()
            v2_flat = v2s.flatten()
            # Normalize by feature range so we can compare across features
            norm = (np.abs(v1_flat - orig1) / (high1 - low1) +
                    np.abs(v2_flat - orig2) / (high2 - low2))
            norm[~flips] = np.inf
            best = np.argmin(norm)
            
            results.append({
                'features': (f1, f2),
                'orig':     (orig1, orig2),
                'new':      (v1_flat[best], v2_flat[best]),
                'changes':  (v1_flat[best] - orig1, v2_flat[best] - orig2),
                'norm':     norm[best],
                'new_prob': probs[best],
            })
    
    results.sort(key=lambda x: x['norm'])
    return results


print(f"\n{'='*80}")
print(f"HARD CASE: pairwise counterfactuals (top 5)")
print(f"{'='*80}")
print("Computing pairwise grid search (~30s)...")

pair_cfs = find_pairwise_cfs(X_val.iloc[[hard_idx]], champion_clf, feature_specs, best_threshold)

if not pair_cfs:
    print("\nNo pair of changes flips this rejection. Need 3+ features simultaneously.")
else:
    for i, cf in enumerate(pair_cfs[:5]):
        f1, f2 = cf['features']
        print(f"\nOption {i+1}: change both {f1} and {f2}")
        print(f"  {f1:25}: {cf['orig'][0]:>10.2f} -> {cf['new'][0]:>10.2f}  ({cf['changes'][0]:+.2f})")
        print(f"  {f2:25}: {cf['orig'][1]:>10.2f} -> {cf['new'][1]:>10.2f}  ({cf['changes'][1]:+.2f})")
        print(f"  P(default): {all_probs[hard_idx]:.4f} -> {cf['new_prob']:.4f}")


HARD CASE: pairwise counterfactuals (top 5)
Computing pairwise grid search (~30s)...


/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/nachimora


Option 1: change both int_rate and annual_inc
  int_rate                 :      13.49 ->       6.79  (-6.70)
  annual_inc               :   32000.00 ->   54285.71  (+22285.71)
  P(default): 0.3128 -> 0.1451

Option 2: change both int_rate and dti
  int_rate                 :      13.49 ->       5.00  (-8.49)
  dti                      :      13.05 ->       8.57  (-4.48)
  P(default): 0.3128 -> 0.1388

Option 3: change both loan_amnt and int_rate
  loan_amnt                :   10000.00 ->    3785.71  (-6214.29)
  int_rate                 :      13.49 ->       5.00  (-8.49)
  P(default): 0.3128 -> 0.1451

Option 4: change both int_rate and fico_range_low
  int_rate                 :      13.49 ->       5.00  (-8.49)
  fico_range_low           :     750.00 ->     796.43  (+46.43)
  P(default): 0.3128 -> 0.1388


/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
